# Chapter 5.8: Scalable Graph Recommendation

## Learning Objectives

By the end of this notebook, you will be able to:

1. Explain why full-batch GNN training is infeasible for billion-scale graphs
2. Implement **neighbor sampling** for mini-batch GNN training (GraphSAGE-style)
3. Understand **layer-wise sampling**, **subgraph sampling**, and **cluster sampling**
4. Implement the **PinSage** approach: random walks + importance-based neighbor sampling
5. Design efficient **mini-batch GNN training pipelines** for recommendation
6. Understand challenges and solutions for **distributed GNN training**
7. Benchmark sampling strategies on training speed and recommendation quality

## Prerequisites

- Chapter 5.5 (LightGCN, graph-based rec)
- Understanding of mini-batch training and sampling
- Basic familiarity with graph data structures

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/rec_system/blob/main/notebooks/part5/chapter_5.8_scalable_graph.ipynb)
[![Download](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/rec_system/main/notebooks/part5/chapter_5.8_scalable_graph.ipynb)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import random
import time
from collections import defaultdict
import scipy.sparse as sp

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cpu")
print(f"Using device: {device}")

## 1. The Scalability Challenge

Industrial recommendation systems deal with **massive** graphs:

| System | Users | Items | Edges |
|--------|-------|-------|-------|
| Pinterest (PinSage) | 1B+ | 2B+ pins | 18B+ |
| Alibaba (Aligraph) | 500M+ | 1B+ | 10B+ |
| YouTube | 2B+ | 800M+ videos | 100B+ |

Full-batch GCN is infeasible because:
1. **Memory**: Storing all embeddings at all layers for $N$ nodes ($O(NLd)$)
2. **Computation**: Each layer multiplies $N \times N$ sparse matrix ($O(|E| \cdot d)$)
3. **Neighbor explosion**: $L$ layers means each node's computation depends on $O(k^L)$ neighbors

> **💡 Concept:** The "neighbor explosion" problem: with $k$ neighbors per node and $L$ layers, computing one node's embedding requires touching $k^L$ nodes. For $k=50, L=3$, that is 125,000 nodes per target node!

## 2. Synthetic Large-Scale Graph

In [ ]:
def generate_large_graph(n_users=2000, n_items=1000, n_interactions=30000, seed=42):
    """Generate a moderately large bipartite graph for scalability experiments."""
    rng = np.random.RandomState(seed)
    
    # Power-law degree distribution
    user_activity = rng.power(0.6, size=n_users)
    user_activity = user_activity / user_activity.sum() * n_interactions
    item_popularity = rng.power(0.4, size=n_items)
    item_popularity /= item_popularity.sum()
    
    interactions = set()
    while len(interactions) < n_interactions:
        u = rng.choice(n_users, p=user_activity / user_activity.sum())
        i = rng.choice(n_items, p=item_popularity)
        interactions.add((int(u), int(i)))
    
    interactions = list(interactions)
    users = [x[0] for x in interactions]
    items = [x[1] for x in interactions]
    
    # Build adjacency list
    adj_list = defaultdict(list)  # node_id -> [neighbor_ids]
    for u, i in zip(users, items):
        adj_list[u].append(n_users + i)
        adj_list[n_users + i].append(u)
    
    user_items = defaultdict(set)
    for u, i in interactions:
        user_items[u].add(i)
    
    return {
        "users": users,
        "items": items,
        "n_users": n_users,
        "n_items": n_items,
        "adj_list": dict(adj_list),
        "user_items": user_items,
        "interactions": interactions,
    }

graph_data = generate_large_graph()
N_USERS = graph_data["n_users"]
N_ITEMS = graph_data["n_items"]
N_NODES = N_USERS + N_ITEMS

print(f"Nodes: {N_NODES} (users: {N_USERS}, items: {N_ITEMS})")
print(f"Edges: {len(graph_data['interactions'])}")

# Degree statistics
degrees = [len(graph_data["adj_list"].get(n, [])) for n in range(N_NODES)]
print(f"Degree — mean: {np.mean(degrees):.1f}, max: {max(degrees)}, median: {np.median(degrees):.0f}")

In [ ]:
# Visualize degree distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

user_degrees = [len(graph_data["adj_list"].get(u, [])) for u in range(N_USERS)]
item_degrees = [len(graph_data["adj_list"].get(N_USERS + i, [])) for i in range(N_ITEMS)]

axes[0].hist(user_degrees, bins=50, edgecolor="black", alpha=0.7, color="steelblue")
axes[0].set_xlabel("Degree")
axes[0].set_ylabel("Count")
axes[0].set_title("User Degree Distribution")
axes[0].set_yscale("log")

axes[1].hist(item_degrees, bins=50, edgecolor="black", alpha=0.7, color="coral")
axes[1].set_xlabel("Degree")
axes[1].set_ylabel("Count")
axes[1].set_title("Item Degree Distribution")
axes[1].set_yscale("log")

plt.tight_layout()
plt.show()

## 3. Neighbor Sampling (GraphSAGE-Style)

**Node-wise neighbor sampling** (Hamilton et al., NeurIPS 2017) limits the number of neighbors per node:

1. For each target node, uniformly sample $k$ neighbors
2. For each sampled neighbor, sample $k$ of *their* neighbors (for the next layer)
3. Continue for $L$ layers

This bounds the computation per mini-batch to $O(B \cdot k^L \cdot d)$ where $B$ is batch size.

> **⚠️ Common Pitfall:** Sampling too few neighbors ($k < 5$) introduces high variance in gradient estimates. Sampling too many ($k > 25$) reduces the scalability benefit. $k \in [10, 25]$ per layer is typical.

In [ ]:
class NeighborSampler:
    """Sample fixed number of neighbors per node for each GCN layer."""
    
    def __init__(self, adj_list, n_nodes):
        self.adj_list = adj_list
        self.n_nodes = n_nodes
    
    def sample(self, target_nodes, num_samples_per_layer):
        """
        Sample multi-hop neighborhoods for a batch of target nodes.
        
        target_nodes: list of node IDs to compute embeddings for
        num_samples_per_layer: list of ints [k_L, k_{L-1}, ...k_1]
                               (from outermost to innermost layer)
        
        Returns: list of (sampled_nodes, edge_index) per layer
        """
        layers = []
        current_nodes = list(target_nodes)
        all_nodes = set(current_nodes)
        
        for k in num_samples_per_layer:
            new_nodes = set()
            edges_src = []  # Source (neighbor)
            edges_dst = []  # Destination (current node)
            
            for node in current_nodes:
                neighbors = self.adj_list.get(node, [])
                if len(neighbors) == 0:
                    continue
                
                if len(neighbors) <= k:
                    sampled = neighbors
                else:
                    sampled = random.sample(neighbors, k)
                
                for n in sampled:
                    edges_src.append(n)
                    edges_dst.append(node)
                    new_nodes.add(n)
            
            layers.append({
                "src": edges_src,
                "dst": edges_dst,
            })
            all_nodes.update(new_nodes)
            current_nodes = list(new_nodes)
        
        # Reindex: map global node IDs to local indices
        all_nodes = sorted(all_nodes)
        node_to_local = {n: i for i, n in enumerate(all_nodes)}
        
        reindexed_layers = []
        for layer in layers:
            local_src = [node_to_local[n] for n in layer["src"]]
            local_dst = [node_to_local[n] for n in layer["dst"]]
            reindexed_layers.append({
                "src": torch.tensor(local_src, dtype=torch.long),
                "dst": torch.tensor(local_dst, dtype=torch.long),
            })
        
        target_local = [node_to_local[n] for n in target_nodes]
        
        return {
            "all_nodes": torch.tensor(all_nodes, dtype=torch.long),
            "target_local": torch.tensor(target_local, dtype=torch.long),
            "layers": reindexed_layers,
        }

# Test
sampler = NeighborSampler(graph_data["adj_list"], N_NODES)
target = [0, 1, 2, 3, 4]
batch = sampler.sample(target, num_samples_per_layer=[10, 10])

print(f"Target nodes: {target}")
print(f"All nodes in subgraph: {batch['all_nodes'].shape[0]}")
print(f"Layer 0 edges: {batch['layers'][0]['src'].shape[0]}")
print(f"Layer 1 edges: {batch['layers'][1]['src'].shape[0]}")

## 4. Mini-Batch GCN with Neighbor Sampling

In [ ]:
class MiniBatchGCN(nn.Module):
    """GCN that supports mini-batch training with neighbor sampling."""
    
    def __init__(self, n_nodes, embed_dim=64, n_layers=2):
        super().__init__()
        self.n_nodes = n_nodes
        self.n_layers = n_layers
        self.embeddings = nn.Embedding(n_nodes, embed_dim)
        nn.init.xavier_uniform_(self.embeddings.weight)
    
    def forward(self, sampled_batch):
        """
        Mini-batch forward pass using sampled neighborhoods.
        sampled_batch: output of NeighborSampler.sample()
        Returns: embeddings for target nodes
        """
        all_node_ids = sampled_batch["all_nodes"]
        target_local = sampled_batch["target_local"]
        layers = sampled_batch["layers"]
        
        # Get initial embeddings for all nodes in subgraph
        h = self.embeddings(all_node_ids)  # (n_subgraph_nodes, embed_dim)
        
        # Note: layers are in reverse order (outermost first)
        # We process from innermost (last) to outermost (first)
        for layer_info in reversed(layers):
            src = layer_info["src"]  # Neighbors
            dst = layer_info["dst"]  # Target nodes
            
            # Message passing: aggregate neighbor embeddings
            src_emb = h[src]  # (n_edges, dim)
            
            # Scatter-mean aggregation
            n_nodes_local = h.size(0)
            agg = torch.zeros(n_nodes_local, h.size(1), device=h.device)
            count = torch.zeros(n_nodes_local, 1, device=h.device)
            agg.scatter_add_(0, dst.unsqueeze(-1).expand_as(src_emb), src_emb)
            count.scatter_add_(0, dst.unsqueeze(-1), torch.ones(dst.size(0), 1))
            count = count.clamp(min=1)
            
            # LightGCN-style: just average neighbors
            h = h + agg / count
        
        # Return embeddings for target nodes only
        return h[target_local]

# Test
mini_gcn = MiniBatchGCN(N_NODES, embed_dim=64, n_layers=2)
out = mini_gcn(batch)
print(f"Mini-batch GCN output shape: {out.shape}")

## 5. Training Pipeline

In [ ]:
def train_minibatch_gcn(model, sampler, interactions, user_items, n_users, n_items,
                        n_epochs=10, batch_size=256, num_neighbors=[10, 10], lr=0.001):
    """Train mini-batch GCN with BPR loss and neighbor sampling."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"loss": [], "time": []}
    
    for epoch in range(n_epochs):
        model.train()
        random.shuffle(interactions)
        total_loss = 0
        n_batches = 0
        epoch_start = time.time()
        
        for start in range(0, len(interactions), batch_size):
            batch_interactions = interactions[start:start + batch_size]
            
            # Collect user, pos_item, neg_item node IDs
            user_nodes = []
            pos_item_nodes = []
            neg_item_nodes = []
            
            for u, pos_i in batch_interactions:
                neg_i = random.randint(0, n_items - 1)
                while neg_i in user_items[u]:
                    neg_i = random.randint(0, n_items - 1)
                user_nodes.append(u)
                pos_item_nodes.append(n_users + pos_i)
                neg_item_nodes.append(n_users + neg_i)
            
            # Sample neighborhoods for all nodes we need
            all_target = list(set(user_nodes + pos_item_nodes + neg_item_nodes))
            sampled = sampler.sample(all_target, num_neighbors)
            
            # Forward
            embeddings = model(sampled)  # (n_targets, dim)
            
            # Map to local indices
            target_to_local = {n: i for i, n in enumerate(all_target)}
            
            u_idx = torch.tensor([target_to_local[n] for n in user_nodes])
            pos_idx = torch.tensor([target_to_local[n] for n in pos_item_nodes])
            neg_idx = torch.tensor([target_to_local[n] for n in neg_item_nodes])
            
            u_emb = embeddings[u_idx]
            pos_emb = embeddings[pos_idx]
            neg_emb = embeddings[neg_idx]
            
            pos_score = (u_emb * pos_emb).sum(-1)
            neg_score = (u_emb * neg_emb).sum(-1)
            loss = -torch.log(torch.sigmoid(pos_score - neg_score) + 1e-8).mean()
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            
            total_loss += loss.item()
            n_batches += 1
        
        epoch_time = time.time() - epoch_start
        avg_loss = total_loss / n_batches
        history["loss"].append(avg_loss)
        history["time"].append(epoch_time)
        
        if (epoch + 1) % 3 == 0:
            print(f"Epoch {epoch+1}/{n_epochs} — Loss: {avg_loss:.4f}, Time: {epoch_time:.1f}s")
    
    return history

# Train
torch.manual_seed(SEED)
model_sampled = MiniBatchGCN(N_NODES, embed_dim=64, n_layers=2)

history_sampled = train_minibatch_gcn(
    model_sampled, sampler, graph_data["interactions"],
    graph_data["user_items"], N_USERS, N_ITEMS,
    n_epochs=10, batch_size=256, num_neighbors=[10, 10]
)

## 6. PinSage: Random Walk + Importance Sampling

**PinSage** (Ying et al., KDD 2018) — developed at Pinterest for 2 billion pins:

1. **Random walk sampling**: Instead of uniform neighbor sampling, use random walks to define neighborhoods
2. **Importance pooling**: Weight neighbors by their visit count in random walks (L1 normalized)
3. **Hard negative sampling**: Use random walks to find "hard" negatives

Random walk advantage: naturally captures multi-hop context and weights neighbors by structural importance.

> **🔑 Pro Tip:** PinSage's random walk sampling has lower variance than uniform sampling because it concentrates on structurally important neighbors. Nodes visited more frequently in random walks contribute more to the target's representation.

In [ ]:
class PinSageSampler:
    """PinSage-style neighbor sampling via random walks."""
    
    def __init__(self, adj_list, n_nodes):
        self.adj_list = adj_list
        self.n_nodes = n_nodes
    
    def random_walk(self, start_node, walk_length=10, n_walks=5):
        """Perform random walks from start_node and return visit counts."""
        visit_count = defaultdict(int)
        
        for _ in range(n_walks):
            current = start_node
            for _ in range(walk_length):
                neighbors = self.adj_list.get(current, [])
                if not neighbors:
                    break
                current = random.choice(neighbors)
                if current != start_node:
                    visit_count[current] += 1
        
        return visit_count
    
    def sample_neighbors(self, target_nodes, k=15, walk_length=10, n_walks=5):
        """
        Sample top-k neighbors by random walk visit frequency.
        Returns: (neighbor_ids, importance_weights) per target node.
        """
        results = []
        for node in target_nodes:
            visits = self.random_walk(node, walk_length, n_walks)
            if not visits:
                results.append(([], []))
                continue
            
            # Top-k by visit count
            sorted_neighbors = sorted(visits.items(), key=lambda x: -x[1])
            top_k = sorted_neighbors[:k]
            
            neighbor_ids = [n for n, _ in top_k]
            counts = np.array([c for _, c in top_k], dtype=np.float32)
            weights = counts / counts.sum()  # L1 normalized
            
            results.append((neighbor_ids, weights.tolist()))
        
        return results
    
    def sample_hard_negatives(self, target_nodes, positive_items, n_neg=5,
                              walk_length=5, n_walks=3):
        """Sample hard negatives: items reachable by random walk but not interacted."""
        hard_negs = []
        for node, pos_set in zip(target_nodes, positive_items):
            visits = self.random_walk(node, walk_length, n_walks)
            candidates = [(n, c) for n, c in visits.items()
                         if n not in pos_set and n != node]
            candidates.sort(key=lambda x: -x[1])
            
            # Middle-ranked items (not too easy, not too hard)
            mid = len(candidates) // 3
            pool = candidates[mid:mid + n_neg * 3] if candidates else []
            if pool:
                sampled = random.sample(pool, min(n_neg, len(pool)))
                hard_negs.append([n for n, _ in sampled])
            else:
                hard_negs.append([])
        
        return hard_negs

# Test
pinsage_sampler = PinSageSampler(graph_data["adj_list"], N_NODES)
target = [0, 1, 2]
neighbors = pinsage_sampler.sample_neighbors(target, k=10)

for i, (node, (nbs, wts)) in enumerate(zip(target, neighbors)):
    print(f"Node {node}: {len(nbs)} neighbors sampled")
    if wts:
        print(f"  Top-3 weights: {wts[:3]}")

In [ ]:
class PinSageConv(nn.Module):
    """Single PinSage convolution layer with importance pooling."""
    
    def __init__(self, embed_dim):
        super().__init__()
        self.W = nn.Linear(embed_dim * 2, embed_dim)
        self.norm = nn.LayerNorm(embed_dim)
    
    def forward(self, node_emb, neighbor_embs, importance_weights):
        """
        node_emb: (batch, dim)
        neighbor_embs: (batch, k, dim)
        importance_weights: (batch, k)
        """
        # Importance-weighted aggregation
        weights = importance_weights.unsqueeze(-1)  # (batch, k, 1)
        agg = (weights * neighbor_embs).sum(dim=1)  # (batch, dim)
        
        # Concatenate and project
        combined = torch.cat([node_emb, agg], dim=-1)  # (batch, 2*dim)
        out = F.relu(self.W(combined))
        out = self.norm(out)
        out = F.normalize(out, dim=-1)  # L2 normalize
        
        return out

# Test
conv = PinSageConv(embed_dim=64)
node_e = torch.randn(4, 64)
nb_e = torch.randn(4, 10, 64)
w = F.softmax(torch.randn(4, 10), dim=-1)
out = conv(node_e, nb_e, w)
print(f"PinSage conv output: {out.shape}")

## 7. Comparing Sampling Strategies

Let's compare the efficiency of different neighbor counts.

In [ ]:
# Benchmark: sampling overhead vs neighbor count
neighbor_counts = [5, 10, 15, 20, 30]
batch_sizes = [64, 128, 256, 512]

# Measure sampling + forward pass time
timing_results = {}

for k in neighbor_counts:
    torch.manual_seed(SEED)
    model_bench = MiniBatchGCN(N_NODES, embed_dim=64, n_layers=2)
    
    target_nodes = list(range(256))
    
    # Measure time
    start = time.time()
    for _ in range(10):  # 10 iterations
        batch = sampler.sample(target_nodes, [k, k])
        with torch.no_grad():
            _ = model_bench(batch)
    elapsed = (time.time() - start) / 10
    
    n_subgraph = batch["all_nodes"].shape[0]
    timing_results[k] = {"time": elapsed, "subgraph_size": n_subgraph}
    print(f"k={k:2d}: {elapsed:.3f}s/iter, subgraph: {n_subgraph} nodes")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ks = list(timing_results.keys())
times = [timing_results[k]["time"] for k in ks]
sizes = [timing_results[k]["subgraph_size"] for k in ks]

axes[0].plot(ks, times, marker="o", color="steelblue")
axes[0].set_xlabel("Neighbors per Layer (k)")
axes[0].set_ylabel("Time per Iteration (s)")
axes[0].set_title("Sampling + Forward Pass Time")
axes[0].grid(True, alpha=0.3)

axes[1].plot(ks, sizes, marker="s", color="coral")
axes[1].set_xlabel("Neighbors per Layer (k)")
axes[1].set_ylabel("Subgraph Size (nodes)")
axes[1].set_title("Subgraph Size vs Neighbor Count")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Cluster Sampling

**Cluster-GCN** (Chiang et al., KDD 2019) partitions the graph into clusters and trains on one cluster at a time:

1. Pre-partition graph using METIS or random clustering
2. Each mini-batch = one or a few clusters
3. Only within-cluster edges are used

Advantage: no neighbor explosion (subgraph is bounded). Disadvantage: inter-cluster edges are ignored.

In [ ]:
class RandomClusterSampler:
    """Simple random graph clustering for Cluster-GCN style training."""
    
    def __init__(self, n_nodes, n_clusters=20):
        self.n_nodes = n_nodes
        self.n_clusters = n_clusters
        # Random assignment (in practice, use METIS for better clustering)
        self.cluster_assignment = np.random.randint(0, n_clusters, size=n_nodes)
        self.clusters = defaultdict(list)
        for node in range(n_nodes):
            self.clusters[self.cluster_assignment[node]].append(node)
    
    def get_cluster_subgraph(self, cluster_ids, adj_list):
        """
        Get the subgraph induced by one or more clusters.
        Returns nodes and intra-cluster edges.
        """
        nodes = set()
        for cid in cluster_ids:
            nodes.update(self.clusters[cid])
        
        node_list = sorted(nodes)
        node_set = set(node_list)
        node_to_local = {n: i for i, n in enumerate(node_list)}
        
        # Collect intra-cluster edges
        src, dst = [], []
        for node in node_list:
            for neighbor in adj_list.get(node, []):
                if neighbor in node_set:
                    src.append(node_to_local[neighbor])
                    dst.append(node_to_local[node])
        
        return {
            "nodes": torch.tensor(node_list, dtype=torch.long),
            "src": torch.tensor(src, dtype=torch.long),
            "dst": torch.tensor(dst, dtype=torch.long),
            "n_nodes": len(node_list),
        }

cluster_sampler = RandomClusterSampler(N_NODES, n_clusters=20)
cluster_sizes = [len(cluster_sampler.clusters[i]) for i in range(20)]
print(f"Cluster sizes — mean: {np.mean(cluster_sizes):.0f}, "
      f"min: {min(cluster_sizes)}, max: {max(cluster_sizes)}")

# Get a sample cluster subgraph
subgraph = cluster_sampler.get_cluster_subgraph([0, 1], graph_data["adj_list"])
print(f"\nCluster [0,1] subgraph: {subgraph['n_nodes']} nodes, {len(subgraph['src'])} edges")

## 9. Industrial-Scale Considerations

Beyond algorithmic sampling, industrial systems face additional challenges:

### Distributed GNN Training

For billion-node graphs:
1. **Graph partitioning**: Split graph across machines (vertex-cut or edge-cut)
2. **Communication**: Fetch remote node features for cross-partition edges
3. **Caching**: Cache frequently accessed node features (hot nodes)

### Serving

PinSage's serving pipeline:
1. Pre-compute embeddings for all items (offline, MapReduce)
2. Build approximate nearest neighbor (ANN) index
3. Real-time: compute user embedding -> ANN query -> candidates

> **🔑 Pro Tip:** In production, GNN embeddings are typically pre-computed offline and stored. Real-time inference uses cached embeddings + a fast ANN index. Full GNN forward pass at serving time is rarely feasible.

In [ ]:
# Simulate the PinSage serving pipeline
print("PinSage Serving Pipeline Simulation")
print("=" * 50)

# Step 1: Pre-compute all item embeddings (offline)
start = time.time()
model_sampled.eval()
all_item_nodes = list(range(N_USERS, N_NODES))  # Item node IDs
batch_size = 256
item_embeddings = []

with torch.no_grad():
    for i in range(0, len(all_item_nodes), batch_size):
        batch_nodes = all_item_nodes[i:i + batch_size]
        batch_sampled = sampler.sample(batch_nodes, [10, 10])
        embs = model_sampled(batch_sampled)
        item_embeddings.append(embs)

item_embeddings = torch.cat(item_embeddings, dim=0)  # (n_items, dim)
item_embeddings = F.normalize(item_embeddings, dim=-1)

precompute_time = time.time() - start
print(f"Step 1: Pre-computed {N_ITEMS} item embeddings in {precompute_time:.2f}s")

# Step 2: ANN index (brute-force for demo)
print(f"Step 2: Built embedding index ({item_embeddings.shape})")

# Step 3: Real-time user query
test_user = 42
start = time.time()

with torch.no_grad():
    user_batch = sampler.sample([test_user], [10, 10])
    user_emb = model_sampled(user_batch)
    user_emb = F.normalize(user_emb, dim=-1)
    
    # ANN search (cosine similarity)
    scores = user_emb @ item_embeddings.T  # (1, n_items)
    _, top_items = scores.topk(10, dim=-1)

query_time = time.time() - start
print(f"Step 3: User query in {query_time * 1000:.1f}ms")
print(f"  Top-10 recommendations for user {test_user}: {top_items[0].tolist()}")

## 10. Exercises

### 🏋️ Exercise 1: Implement Mini-Batch GNN Training with Neighbor Sampling

In [ ]:
# 🏋️ Exercise 1: Full Mini-Batch Training Pipeline
#
# Implement a complete training pipeline with:
# 1. Neighbor sampling at each GCN layer
# 2. BPR loss for recommendation
# 3. Evaluation with HR@20
# 4. Compare different neighbor counts: k=5, 10, 20

def full_minibatch_pipeline(k_neighbors, n_epochs=8):
    # TODO: Implement
    # 1. Create model, sampler, optimizer
    # 2. Train loop with neighbor sampling
    # 3. Evaluate on held-out test set
    # 4. Return HR@20 and training time
    pass

# for k in [5, 10, 20]:
#     hr, time = full_minibatch_pipeline(k)
#     print(f"k={k}: HR@20={hr:.4f}, Time={time:.1f}s")

### 🏋️ Exercise 2: Implement PinSage-Style Hard Negative Mining

In [ ]:
# 🏋️ Exercise 2: Hard Negative Mining
#
# Implement PinSage-style hard negative sampling:
# 1. Use random walks to find items reachable from user but not interacted
# 2. These are "hard negatives" (structurally close but not positive)
# 3. Train with a mix of random negatives + hard negatives
# 4. Compare with random-only negative sampling

def train_with_hard_negatives(model, sampler, pinsage_sampler, interactions,
                               user_items, n_users, n_items, n_epochs=8):
    # TODO: Implement
    # For each batch:
    # 1. Sample random negatives (standard)
    # 2. Sample hard negatives via PinSageSampler.sample_hard_negatives()
    # 3. Compute BPR loss for both types
    # 4. Total loss = loss_random + lambda * loss_hard
    pass

### 🏋️ Exercise 3: Cluster-GCN Training

In [ ]:
# 🏋️ Exercise 3: Cluster-GCN Training
#
# Implement Cluster-GCN style training:
# 1. Partition the graph into clusters
# 2. Each training step: sample 2-3 clusters
# 3. Run GCN on the subgraph induced by these clusters
# 4. Compute BPR loss only for edges within the subgraph
# 5. Compare training speed and quality with neighbor sampling

class ClusterGCN(nn.Module):
    def __init__(self, n_nodes, embed_dim=64, n_layers=2):
        super().__init__()
        # TODO: Implement
        # Same as MiniBatchGCN but operates on cluster subgraphs
        pass
    
    def forward(self, cluster_subgraph):
        # TODO: Implement GCN on subgraph
        # 1. Get embeddings for nodes in subgraph
        # 2. Run message passing using intra-cluster edges
        # 3. Return updated embeddings
        pass

# TODO: Train and compare with neighbor sampling

## Summary

| Strategy | Paper | Complexity per Batch | Key Idea |
|----------|-------|---------------------|----------|
| Full-batch | GCN | $O(|E| \cdot d)$ | Process entire graph |
| Node-wise sampling | GraphSAGE | $O(B \cdot k^L \cdot d)$ | Sample k neighbors per layer |
| Cluster sampling | Cluster-GCN | $O(|E_c| \cdot d)$ | Process cluster subgraphs |
| Random walk | PinSage | $O(B \cdot k \cdot d)$ | Importance-based neighbor selection |
| Layer-wise | FastGCN | $O(B \cdot s \cdot d)$ | Sample s nodes per layer |

**Key Takeaways:**
1. Full-batch GNN training is infeasible beyond ~100K nodes
2. Neighbor sampling (GraphSAGE) is the most common approach: simple and effective
3. PinSage's random walk sampling achieves importance-aware neighbor selection
4. Hard negative mining significantly improves recommendation quality
5. In production, pre-compute embeddings offline and serve with ANN indices
6. Distributed training is essential for billion-scale graphs (graph partitioning + communication)